# Dukascopy長期データ取得と環境調整

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 25


In [ ]:
!pip install dukascopy-python

## 元のセル index 26


In [ ]:
# ============================================================
# USD/JPY LONG-HISTORY DATASET BUILDER
#
# 目的
# ------------------------------------------------------------
# 10年程度のUSD/JPYデータをDukascopyから取得し、
#
# 5分足
# 15分足
# 30分足
#
# を同じ期間で保存する。
#
# 次の実験で
# 「5分足はノイズを過学習しているのか？」
# を検証するためのデータ作成コード。
# ============================================================


from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

import dukascopy_python


# ============================================================
# 1. 設定
# ============================================================

# 10年間
START_DATE = datetime(
    2016,
    1,
    1,
    tzinfo=timezone.utc
)

END_DATE = datetime(
    2026,
    9,
    1,
    tzinfo=timezone.utc
)

OUTPUT_DIR = (
    Path.cwd()
    / "dukascopy_usdjpy"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


# ============================================================
# 2. USD/JPY instrumentを探す
# ============================================================

# dukascopy-python のバージョンによって
# 定数名が異なる可能性があるので
# まず候補を探す

instrument_candidates = [
    name
    for name
    in dir(
        dukascopy_python.instruments
    )
    if (
        "USD" in name.upper()
        and
        "JPY" in name.upper()
    )
]


print(
    "USD/JPY instrument候補:"
)

for name in instrument_candidates:
    print(
        name
    )


# ============================================================
# 3. USD/JPY定数を取得
# ============================================================

preferred_names = [
    "INSTRUMENT_FX_MAJORS_USD_JPY",
    "INSTRUMENT_FX_USD_JPY",
    "INSTRUMENT_USD_JPY",
]


instrument = None


for name in preferred_names:

    if hasattr(
        dukascopy_python.instruments,
        name
    ):

        instrument = getattr(
            dukascopy_python.instruments,
            name
        )

        print()
        print(
            "使用instrument:",
            name
        )

        break


if instrument is None:

    # 候補が1つだけなら自動採用
    if len(
        instrument_candidates
    ) == 1:

        name = (
            instrument_candidates[
                0
            ]
        )

        instrument = getattr(
            dukascopy_python.instruments,
            name
        )

        print()
        print(
            "自動選択instrument:",
            name
        )

    else:

        raise RuntimeError(
            "USD/JPY instrumentを自動判定できませんでした。"
            "上に表示された候補名を確認してください。"
        )


# ============================================================
# 4. interval候補を確認
# ============================================================

interval_candidates = [
    name
    for name
    in dir(
        dukascopy_python
    )
    if (
        name.startswith(
            "INTERVAL_"
        )
    )
]


print()
print(
    "利用可能interval候補:"
)

for name in interval_candidates:
    print(
        name
    )


# ============================================================
# 5. interval定数取得
# ============================================================

def get_interval_constant(
    candidates
):

    for name in candidates:

        if hasattr(
            dukascopy_python,
            name
        ):

            return (
                name,
                getattr(
                    dukascopy_python,
                    name
                )
            )

    return (
        None,
        None
    )


name_5m, interval_5m = (
    get_interval_constant(
        [
            "INTERVAL_MIN_5",
            "INTERVAL_MINUTES_5",
        ]
    )
)


name_15m, interval_15m = (
    get_interval_constant(
        [
            "INTERVAL_MIN_15",
            "INTERVAL_MINUTES_15",
        ]
    )
)


name_30m, interval_30m = (
    get_interval_constant(
        [
            "INTERVAL_MIN_30",
            "INTERVAL_MINUTES_30",
        ]
    )
)


print()
print(
    "5分:",
    name_5m
)

print(
    "15分:",
    name_15m
)

print(
    "30分:",
    name_30m
)


if (
    interval_5m is None
    or
    interval_15m is None
    or
    interval_30m is None
):

    raise RuntimeError(
        "必要なinterval定数が見つかりません。"
        "上のinterval一覧を確認してください。"
    )


# ============================================================
# 6. Bid側を使用
# ============================================================

if hasattr(
    dukascopy_python,
    "OFFER_SIDE_BID"
):

    OFFER_SIDE = (
        dukascopy_python.OFFER_SIDE_BID
    )

else:

    raise RuntimeError(
        "OFFER_SIDE_BID が見つかりません。"
    )


# ============================================================
# 7. データ取得関数
# ============================================================

def fetch_usdjpy(
    interval,
    interval_name,
):

    print()
    print(
        "===================================="
    )

    print(
        interval_name,
        "取得開始"
    )

    print(
        "===================================="
    )

    print(
        "期間:",
        START_DATE,
        "→",
        END_DATE
    )

    df = dukascopy_python.fetch(

        instrument=
            instrument,

        interval=
            interval,

        offer_side=
            OFFER_SIDE,

        start=
            START_DATE,

        end=
            END_DATE,
    )


    if (
        df is None
        or
        len(df) == 0
    ):

        raise RuntimeError(
            f"{interval_name} の取得結果が空です"
        )


    df = (
        df.copy()
    )


    print(
        "取得行数:",
        len(df)
    )

    print(
        "columns:",
        list(
            df.columns
        )
    )

    print(
        "index:",
        df.index[
            0
        ],
        "→",
        df.index[
            -1
        ]
    )


    return df


# ============================================================
# 8. まず15分足を取得
#
# 最初に15分で通信・API仕様を確認する
# ============================================================

df_15m = (
    fetch_usdjpy(
        interval_15m,
        "15分足"
    )
)


print()
print(
    df_15m.head()
)


# ============================================================
# 9. OHLC形式を標準化
# ============================================================

def normalize_ohlc(
    df
):

    frame = (
        df.copy()
    )


    # --------------------------------
    # indexをDatetimeIndexへ
    # --------------------------------

    if not isinstance(
        frame.index,
        pd.DatetimeIndex
    ):

        frame.index = (
            pd.to_datetime(
                frame.index,
                utc=True
            )
        )


    if (
        frame.index.tz
        is None
    ):

        frame.index = (
            frame.index
            .tz_localize(
                "UTC"
            )
        )


    frame.index = (
        frame.index
        .tz_convert(
            "UTC"
        )
    )


    # --------------------------------
    # column名を小文字比較
    # --------------------------------

    column_map = {
        str(col).lower():
            col
        for col
        in frame.columns
    }


    required_names = [
        "open",
        "high",
        "low",
        "close",
    ]


    missing = [
        name
        for name
        in required_names
        if name
        not in column_map
    ]


    if missing:

        raise ValueError(
            "OHLC列が見つかりません。"
            f" missing={missing}, "
            f"columns={list(frame.columns)}"
        )


    frame = frame[
        [
            column_map[
                "open"
            ],

            column_map[
                "high"
            ],

            column_map[
                "low"
            ],

            column_map[
                "close"
            ],
        ]
    ].copy()


    frame.columns = [
        "Open",
        "High",
        "Low",
        "Close",
    ]


    # --------------------------------
    # 数値化
    # --------------------------------

    for col in [
        "Open",
        "High",
        "Low",
        "Close",
    ]:

        frame[
            col
        ] = pd.to_numeric(
            frame[
                col
            ],
            errors="coerce"
        )


    frame = (
        frame
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan
        )
        .dropna()
    )


    # --------------------------------
    # 重複除去
    # --------------------------------

    frame = (
        frame[
            ~frame.index
            .duplicated(
                keep="first"
            )
        ]
    )


    frame = (
        frame.sort_index()
    )


    # --------------------------------
    # OHLC整合性
    # --------------------------------

    valid = (
        (
            frame[
                "High"
            ]
            >=
            frame[
                [
                    "Open",
                    "Close",
                    "Low",
                ]
            ].max(
                axis=1
            )
        )
        &
        (
            frame[
                "Low"
            ]
            <=
            frame[
                [
                    "Open",
                    "Close",
                    "High",
                ]
            ].min(
                axis=1
            )
        )
    )


    frame = (
        frame.loc[
            valid
        ]
        .copy()
    )


    return frame


# ============================================================
# 10. 15分足標準化
# ============================================================

df_15m = (
    normalize_ohlc(
        df_15m
    )
)


print()
print(
    "15分足標準化後:",
    len(
        df_15m
    )
)

print(
    df_15m.head()
)


# ============================================================
# 11. 5分足取得
# ============================================================

df_5m = (
    fetch_usdjpy(
        interval_5m,
        "5分足"
    )
)


df_5m = (
    normalize_ohlc(
        df_5m
    )
)


print(
    "5分足標準化後:",
    len(
        df_5m
    )
)


# ============================================================
# 12. 30分足取得
# ============================================================

df_30m = (
    fetch_usdjpy(
        interval_30m,
        "30分足"
    )
)


df_30m = (
    normalize_ohlc(
        df_30m
    )
)


print(
    "30分足標準化後:",
    len(
        df_30m
    )
)


# ============================================================
# 13. 共通期間へ切り揃える
# ============================================================

common_start = max(
    df_5m.index.min(),
    df_15m.index.min(),
    df_30m.index.min(),
)


common_end = min(
    df_5m.index.max(),
    df_15m.index.max(),
    df_30m.index.max(),
)


print()
print(
    "共通期間:"
)

print(
    common_start,
    "→",
    common_end
)


df_5m = (
    df_5m.loc[
        common_start:
        common_end
    ]
)


df_15m = (
    df_15m.loc[
        common_start:
        common_end
    ]
)


df_30m = (
    df_30m.loc[
        common_start:
        common_end
    ]
)


# ============================================================
# 14. CSV保存
# ============================================================

path_5m = (
    OUTPUT_DIR
    / "usdjpy_5m_2016_2026.csv"
)


path_15m = (
    OUTPUT_DIR
    / "usdjpy_15m_2016_2026.csv"
)


path_30m = (
    OUTPUT_DIR
    / "usdjpy_30m_2016_2026.csv"
)


df_5m.to_csv(
    path_5m,
    index_label=
        "timestamp"
)


df_15m.to_csv(
    path_15m,
    index_label=
        "timestamp"
)


df_30m.to_csv(
    path_30m,
    index_label=
        "timestamp"
)


# ============================================================
# 15. 基本統計
# ============================================================

summary = pd.DataFrame(
    [
        {
            "timeframe":
                "5m",

            "rows":
                len(
                    df_5m
                ),

            "start":
                df_5m.index.min(),

            "end":
                df_5m.index.max(),
        },

        {
            "timeframe":
                "15m",

            "rows":
                len(
                    df_15m
                ),

            "start":
                df_15m.index.min(),

            "end":
                df_15m.index.max(),
        },

        {
            "timeframe":
                "30m",

            "rows":
                len(
                    df_30m
                ),

            "start":
                df_30m.index.min(),

            "end":
                df_30m.index.max(),
        },
    ]
)


print()
print(
    "===================================="
)

print(
    "データセット完成"
)

print(
    "===================================="
)


print(
    summary.to_string(
        index=False
    )
)


print()
print(
    "保存先:"
)

print(
    path_5m.resolve()
)

print(
    path_15m.resolve()
)

print(
    path_30m.resolve()
)


# ============================================================
# 16. 欠損間隔診断
# ============================================================

def gap_diagnostics(
    df,
    expected_minutes,
):

    gaps = (
        df.index
        .to_series()
        .diff()
    )


    expected = (
        pd.Timedelta(
            minutes=
                expected_minutes
        )
    )


    abnormal = (
        gaps[
            gaps
            > expected
        ]
    )


    print()
    print(
        expected_minutes,
        "分足"
    )

    print(
        "通常より長いgap:",
        len(
            abnormal
        )
    )


    print(
        "最大gap:",
        abnormal.max()
        if len(
            abnormal
        )
        else None
    )


gap_diagnostics(
    df_5m,
    5
)


gap_diagnostics(
    df_15m,
    15
)


gap_diagnostics(
    df_30m,
    30
)


print()
print(
    "次の実験では"
)

print(
    "5m / 15m / 30m の"
)

print(
    "Train AUC / Test AUC / Gap / Shuffle / Logistic / RandomForest"
)

print(
    "を同条件で比較します。"
)

## 元のセル index 27


In [ ]:
import dukascopy_python

print("dukascopy_python:")
print(dir(dukascopy_python))

print("\nバージョン情報:")
try:
    print(dukascopy_python.__version__)
except:
    print("version属性なし")

print("\nfetch:")
try:
    import inspect
    print(inspect.signature(dukascopy_python.fetch))
except Exception as e:
    print(e)

## 元のセル index 28


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import dukascopy_python


# ============================================
# 設定
# ============================================

START_DATE = datetime(
    2016,
    1,
    1,
    tzinfo=timezone.utc
)

END_DATE = datetime(
    2026,
    9,
    1,
    tzinfo=timezone.utc
)

INSTRUMENT = "USDJPY"

OUTPUT_DIR = (
    Path.cwd()
    / "dukascopy_usdjpy"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


# ============================================
# 15分足取得
# ============================================

print("15分足を取得します...")

df_15m = dukascopy_python.fetch(
    instrument=INSTRUMENT,
    interval=dukascopy_python.INTERVAL_MIN_15,
    offer_side=dukascopy_python.OFFER_SIDE_BID,
    start=START_DATE,
    end=END_DATE,
)

print()
print("取得完了")
print("行数:", len(df_15m))
print("columns:", df_15m.columns.tolist())

print()
print(df_15m.head())

print()
print(df_15m.tail())


# ============================================
# 保存
# ============================================

save_path = (
    OUTPUT_DIR
    / "usdjpy_15m_2016_2026.csv"
)

df_15m.to_csv(
    save_path,
    index=True
)

print()
print("保存先:")
print(save_path.resolve())

## 元のセル index 29


In [ ]:
!pip uninstall -y dukascopy-python
!pip install --upgrade dukascopy-python

## 元のセル index 30


In [ ]:
from datetime import datetime

import dukascopy_python
from dukascopy_python.instruments import INSTRUMENT_FX_MAJORS_USD_JPY


start = datetime(2026, 8, 1)
end = datetime(2026, 8, 8)


print("USD/JPY 15分足をテスト取得します...")


df = dukascopy_python.fetch(
    instrument=INSTRUMENT_FX_MAJORS_USD_JPY,
    interval=dukascopy_python.INTERVAL_MIN_15,
    offer_side=dukascopy_python.OFFER_SIDE_BID,
    start=start,
    end=end,
    debug=True,
)


print()
print("取得完了")
print("行数:", len(df))
print(df.head())
print(df.tail())

## 元のセル index 31


In [ ]:
# ============================================================
# USD/JPY 15分足 10年分
# 年単位で安全に取得して結合する
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import time

import pandas as pd
import dukascopy_python

from dukascopy_python.instruments import (
    INSTRUMENT_FX_MAJORS_USD_JPY
)


# ============================================================
# 1. 設定
# ============================================================

START_YEAR = 2016
END_YEAR = 2026

OUTPUT_DIR = (
    Path.cwd()
    / "dukascopy_usdjpy"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)

FINAL_PATH = (
    OUTPUT_DIR
    / "usdjpy_15m_2016_2026.csv"
)


# ============================================================
# 2. 1期間取得する関数
# ============================================================

def fetch_period(
    start,
    end,
):

    print(
        f"取得中: {start} → {end}"
    )

    df = dukascopy_python.fetch(
        instrument=
            INSTRUMENT_FX_MAJORS_USD_JPY,

        interval=
            dukascopy_python.INTERVAL_MIN_15,

        offer_side=
            dukascopy_python.OFFER_SIDE_BID,

        start=
            start,

        end=
            end,

        max_retries=
            10,

        debug=
            False,
    )

    if (
        df is None
        or
        len(df) == 0
    ):

        print(
            "取得データなし"
        )

        return pd.DataFrame()

    df = df.copy()

    df.index = pd.to_datetime(
        df.index,
        utc=True
    )

    df = (
        df
        .sort_index()
    )

    return df


# ============================================================
# 3. 年単位で取得
# ============================================================

yearly_frames = []

year_summary = []


for year in range(
    START_YEAR,
    END_YEAR + 1
):

    start = datetime(
        year,
        1,
        1,
        tzinfo=timezone.utc
    )

    if (
        year
        ==
        END_YEAR
    ):

        end = datetime(
            2026,
            9,
            1,
            tzinfo=timezone.utc
        )

    else:

        end = datetime(
            year + 1,
            1,
            1,
            tzinfo=timezone.utc
        )

    try:

        df_year = fetch_period(
            start,
            end
        )

        if (
            not df_year.empty
        ):

            yearly_frames.append(
                df_year
            )

            year_summary.append(
                {
                    "year":
                        year,

                    "rows":
                        len(
                            df_year
                        ),

                    "start":
                        df_year.index.min(),

                    "end":
                        df_year.index.max(),
                }
            )

            # 年ごとにも保存
            year_path = (
                OUTPUT_DIR
                /
                f"usdjpy_15m_{year}.csv"
            )

            df_year.to_csv(
                year_path,
                index=True
            )

            print(
                "完了:",
                year,
                "行数:",
                len(df_year)
            )

        else:

            print(
                "空データ:",
                year
            )

    except Exception as e:

        print()
        print(
            "ERROR:",
            year
        )

        print(
            type(e).__name__,
            e
        )

        print(
            "この年だけ飛ばします。"
        )

    # サーバー負荷を少し下げる
    time.sleep(
        1
    )


# ============================================================
# 4. 結合
# ============================================================

if (
    not yearly_frames
):

    raise RuntimeError(
        "データを1件も取得できませんでした"
    )


all_15m = pd.concat(
    yearly_frames
)


# ============================================================
# 5. 重複削除・時系列整理
# ============================================================

all_15m = (
    all_15m
    [
        ~all_15m.index.duplicated(
            keep="first"
        )
    ]
    .sort_index()
)


# ============================================================
# 6. OHLC異常チェック
# ============================================================

required_columns = [
    "open",
    "high",
    "low",
    "close",
]


missing = [
    col
    for col
    in required_columns
    if col
    not in all_15m.columns
]


if missing:

    raise ValueError(
        f"必要列がありません: {missing}"
    )


invalid_ohlc = (
    (
        all_15m[
            "high"
        ]
        <
        all_15m[
            [
                "open",
                "close",
                "low",
            ]
        ].max(
            axis=1
        )
    )
    |
    (
        all_15m[
            "low"
        ]
        >
        all_15m[
            [
                "open",
                "close",
                "high",
            ]
        ].min(
            axis=1
        )
    )
)


print()
print(
    "OHLC異常行:",
    invalid_ohlc.sum()
)


# ============================================================
# 7. 欠損値確認
# ============================================================

print()
print(
    "欠損値:"
)

print(
    all_15m.isna().sum()
)


# ============================================================
# 8. 時間gap確認
# ============================================================

time_diff = (
    all_15m.index
    .to_series()
    .diff()
)


large_gaps = (
    time_diff
    >
    pd.Timedelta(
        hours=24
    )
)


print()
print(
    "24時間超gap:",
    large_gaps.sum()
)


# ============================================================
# 9. 保存
# ============================================================

all_15m.to_csv(
    FINAL_PATH,
    index=True
)


# ============================================================
# 10. 年別結果
# ============================================================

summary_df = pd.DataFrame(
    year_summary
)


print()
print(
    "===================================="
)

print(
    "年別取得結果"
)

print(
    "===================================="
)

print(
    summary_df.to_string(
        index=False
    )
)


# ============================================================
# 11. 最終結果
# ============================================================

print()
print(
    "===================================="
)

print(
    "15分足10年データ完成"
)

print(
    "===================================="
)

print(
    "総行数:",
    len(
        all_15m
    )
)

print(
    "開始:",
    all_15m.index.min()
)

print(
    "終了:",
    all_15m.index.max()
)

print(
    "保存先:"
)

print(
    FINAL_PATH.resolve()
)

print()
print(
    all_15m.head()
)

print()
print(
    all_15m.tail()
)